# 🔮 Foresee Model — *where will incidents happen next?*

Events are aggregated to **(corridor, hour, day-of-week)** cells, then a
**Poisson `HistGradientBoostingRegressor`** forecasts the expected count per cell.
Quality is **precision@k**: of the top-k predicted hotspots, how many actually saw
an incident in the held-out tail.

This **retrains and overwrites** `models/foresee/`.

In [ ]:
# 📦 Setup — locate the backend/ root (folder containing app/ and models/)
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "app").is_dir() and root != root.parent:
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))
print("✅ Environment ready")
print(f"   📂 Working dir : {os.getcwd()}")
print(f"   🐍 Python      : {sys.version.split()[0]}")

## 📦 Imports & configuration

In [ ]:
import json
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

from app.config import settings
from app.engines.foresee.hotspot import CATEGORICAL, make_features

TEST_FRACTION = 0.20
RANDOM_SEED   = 42
PRECISION_KS  = (10, 20, 50)
IGNORED       = frozenset({"non-corridor", "unknown", "none", ""})
_MODEL_DIR    = settings.models_dir / "foresee"

print("✅ Config loaded  ·  🗺️  categorical features:", CATEGORICAL)

## 📥 Step 1 — Keep rows with a real corridor + temporal features

In [ ]:
from app.ml.pipeline import load_clean

print("📥 Loading events and filtering to mapped corridors ...")
df = load_clean()
corridor = df["corridor"].astype("string")
mask = (
    df["start_datetime"].notna() & df["hour_ist"].notna() & df["dow"].notna()
    & corridor.notna() & ~corridor.str.strip().str.lower().isin(IGNORED)
)
data = df.loc[mask].copy().sort_values("start_datetime", kind="mergesort")
corridors = sorted(data["corridor"].astype(str).unique())
encoder = {c: i for i, c in enumerate(corridors)}

print(f"✅ {len(data):,} events across {len(corridors)} corridors")
print("   🛣️ ", ", ".join(corridors[:6]), "…")

## ✂️ Step 2 — Time split & aggregate to (corridor, hour, dow) cells

In [ ]:
cut = int(len(data) * (1 - TEST_FRACTION))
train_df, test_df = data.iloc[:cut], data.iloc[cut:]

def aggregate(frame):
    agg = (frame.groupby(["corridor", "hour_ist", "dow"], observed=True)
                .size().reset_index(name="count"))
    feats = make_features([encoder[c] for c in agg["corridor"]],
                          agg["hour_ist"].astype(int).tolist(),
                          agg["dow"].astype(int).tolist())
    return feats, agg["count"].to_numpy(dtype=float)

x_train, y_train = aggregate(train_df)
print(f"✂️  Train cells: {len(y_train):,}   ·   busiest cell saw {int(y_train.max())} incidents")
x_train.head(3)

## 🌲 Step 3 — Fit the Poisson gradient-boosting forecaster

In [ ]:
print("🌲 Training HistGradientBoostingRegressor (Poisson loss) ...")
# Tuned config (see _tune_foresee.py): lifts precision@10/20/50 0.80/0.75/0.74 → 0.90/0.80/0.76
model = HistGradientBoostingRegressor(
    loss="poisson", categorical_features=CATEGORICAL,
    max_iter=500, learning_rate=0.03, max_depth=5,
    l2_regularization=2.0, min_samples_leaf=30, random_state=RANDOM_SEED,
)
model.fit(x_train, y_train)
print(f"✅ Trained on {len(y_train):,} corridor·hour·dow cells")

## 📊 Step 4 — Rank the full grid & score precision@k

In [ ]:
grid = [(c, h, d) for c in corridors for h in range(24) for d in range(7)]
gp = model.predict(make_features([encoder[c] for c, _, _ in grid],
                                 [h for _, h, _ in grid], [d for _, _, d in grid]))
pred_rank = [k for _, k in sorted(zip(gp, grid), key=lambda t: t[0], reverse=True)]

test_agg = (test_df.groupby(["corridor", "hour_ist", "dow"], observed=True)
                   .size().reset_index(name="count"))
relevant = {(str(r.corridor), int(r.hour_ist), int(r.dow)) for r in test_agg.itertuples()}

metrics = {f"precision_at_{k}": round(len(set(pred_rank[:k]) & relevant) / k, 4) for k in PRECISION_KS}
metrics["test_cells"] = int(len(test_agg))

print("📊 Hotspot ranking quality")
for k in PRECISION_KS:
    bar = "█" * int(metrics[f'precision_at_{k}'] * 20)
    print(f"   precision@{k:<2}: {metrics[f'precision_at_{k}']:.2f}  {bar}")
metrics

## 💾 Step 5 — Centroids, artifacts & registry

In [ ]:
cent = (data[(data["latitude"].notna()) & (data["latitude"] != 0)]
        .groupby("corridor", observed=True)[["latitude", "longitude"]].mean())
centroids = {c: [round(float(r.latitude), 6), round(float(r.longitude), 6)] for c, r in cent.iterrows()}

_MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, _MODEL_DIR / "forecaster.joblib")
(_MODEL_DIR / "corridor_encoder.json").write_text(json.dumps(encoder, indent=2))
(_MODEL_DIR / "centroids.json").write_text(json.dumps(centroids, indent=2))

entry = {
    "model": "foresee",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "estimator": "HistGradientBoostingRegressor(loss=poisson)",
    "target": "incident_count_per_cell",
    "split": {"type": "time-based", "test_fraction": TEST_FRACTION},
    "n_corridors": len(corridors),
    "n_train_cells": int(len(y_train)),
    "metrics": metrics,
    "artifacts": {
        "forecaster": "foresee/forecaster.joblib",
        "corridor_encoder": "foresee/corridor_encoder.json",
        "centroids": "foresee/centroids.json",
    },
}
registry_path = settings.models_dir / "registry.json"
registry = json.loads(registry_path.read_text()) if registry_path.exists() else {}
registry["foresee"] = entry
registry_path.write_text(json.dumps(registry, indent=2))

print(f"💾 Saved forecaster + {len(centroids)} corridor centroids + registry")
print("🏁 Foresee training complete!")